# V3-2 — Sliding-window aligned dataset and MIL bags

This notebook creates reproducible window manifests only. It does not decode frames, train a model or use event/alert times as model features.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

DATA_ROOT = Path(r'P:\NexarCollisionData')
MANIFEST_ROOT = DATA_ROOT / 'manifests_v3'
REPORT_ROOT = DATA_ROOT / 'reports_v3'
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

VIDEO_MANIFEST_V2_PATH = DATA_ROOT / 'video_manifest_v2.csv'
SELECTED_TRAIN_PATH = DATA_ROOT / 'selected_train_600.csv'
CV_FOLDS_PATH = MANIFEST_ROOT / 'cv_folds_v3.csv'
REGISTRY_PATH = REPORT_ROOT / 'experiments_v3_registry.csv'

WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 2.5
NUM_FRAMES = 16
CORE_END_BEFORE_EVENT_SECONDS = 1.5
CORE_END_AFTER_EVENT_SECONDS = 1.0
ALERT_MARGIN_SECONDS = 0.5
AFTERMATH_IGNORE_SECONDS = 3.0
CONTEXT_SOFT_LABEL = 0.7
REVIEW_SEED = 42
REVIEW_WINDOWS_PER_ROLE = 50
EPSILON = 1e-6

for item in [VIDEO_MANIFEST_V2_PATH, SELECTED_TRAIN_PATH, CV_FOLDS_PATH, REGISTRY_PATH]:
    assert item.is_file(), 'Missing prerequisite: {}'.format(item)
print({'window_seconds': WINDOW_SECONDS, 'window_stride_seconds': WINDOW_STRIDE_SECONDS, 'num_frames': NUM_FRAMES, 'data_root': str(DATA_ROOT)})


{'window_seconds': 5.0, 'window_stride_seconds': 2.5, 'num_frames': 16, 'data_root': 'P:\\NexarCollisionData'}


In [2]:
# 1) Build the central V3 video manifest and attach time_of_alert for manifest-only use
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

video_manifest = pd.read_csv(VIDEO_MANIFEST_V2_PATH).copy()
selected_train = pd.read_csv(SELECTED_TRAIN_PATH).copy()
cv_folds = pd.read_csv(CV_FOLDS_PATH).copy()
video_manifest['video_id'] = video_manifest['video_id'].astype(str)
cv_folds['video_id'] = cv_folds['video_id'].astype(str)
selected_train['local_path'] = selected_train['local_path'].astype(str)

alert_table = selected_train[['local_path', 'row_idx', 'time_of_alert']].rename(columns={'local_path': 'video_path'}).copy()
alert_table['time_of_alert'] = pd.to_numeric(alert_table['time_of_alert'], errors='coerce')
assert alert_table['video_path'].is_unique
v3_video_manifest = video_manifest.merge(alert_table, on='video_path', how='left', validate='one_to_one')
v3_video_manifest = v3_video_manifest.merge(cv_folds[['video_id', 'outer_fold']], on='video_id', how='left', validate='one_to_one')
for field in ['label', 'duration', 'time_of_event', 'fps']:
    v3_video_manifest[field] = pd.to_numeric(v3_video_manifest[field], errors='coerce')
v3_video_manifest['time_of_alert_valid'] = np.where(
    v3_video_manifest['label'].eq(1) & v3_video_manifest['time_of_alert'].notna(),
    v3_video_manifest['time_of_alert'].between(0.0, v3_video_manifest['duration']) & v3_video_manifest['time_of_alert'].le(v3_video_manifest['time_of_event']),
    False
)
v3_video_manifest['relative_event_position'] = np.where(v3_video_manifest['label'].eq(1), v3_video_manifest['time_of_event'] / v3_video_manifest['duration'], np.nan)
v3_video_manifest['v3_window_version'] = 'v3_sliding_5s_stride2.5_core_context_v1'
v3_video_manifest['source_video_manifest_sha256'] = sha256_file(VIDEO_MANIFEST_V2_PATH)
v3_video_manifest['source_cv_folds_sha256'] = sha256_file(CV_FOLDS_PATH)

assert len(v3_video_manifest) == 600
assert v3_video_manifest['video_id'].is_unique
assert v3_video_manifest['outer_fold'].notna().all()
assert v3_video_manifest['is_valid'].astype(str).str.lower().eq('true').all()
assert v3_video_manifest.loc[v3_video_manifest['label'].eq(1), 'time_of_event'].between(0.0, v3_video_manifest.loc[v3_video_manifest['label'].eq(1), 'duration']).all()
assert v3_video_manifest.loc[v3_video_manifest['label'].eq(1), 'time_of_alert_valid'].all()
assert v3_video_manifest.loc[v3_video_manifest['label'].eq(0), 'time_of_alert'].isna().all()

VIDEO_MANIFEST_V3_PATH = MANIFEST_ROOT / 'video_manifest_v3.csv'
v3_video_manifest.to_csv(VIDEO_MANIFEST_V3_PATH, index=False)
print({'videos': len(v3_video_manifest), 'positive_alerts_valid': int(v3_video_manifest['time_of_alert_valid'].sum()), 'video_manifest_v3': str(VIDEO_MANIFEST_V3_PATH)})


{'videos': 600, 'positive_alerts_valid': 300, 'video_manifest_v3': 'P:\\NexarCollisionData\\manifests_v3\\video_manifest_v3.csv'}


In [3]:
# 2) Deterministic sliding-window generation and conservative label policy
def sliding_starts(duration: float, window_seconds: float, stride_seconds: float) -> list[float]:
    assert duration >= window_seconds
    last_start = max(0.0, duration - window_seconds)
    starts = list(np.arange(0.0, last_start + EPSILON, stride_seconds, dtype=float))
    if not starts or abs(starts[-1] - last_start) > EPSILON:
        starts.append(last_start)
    return [round(float(start), 6) for start in starts]

def role_for_positive_window(start: float, end: float, event_time: float, alert_time: float) -> tuple[str, float, float, bool]:
    contains_event = start - EPSILON <= event_time <= end + EPSILON
    end_near_event = event_time - CORE_END_BEFORE_EVENT_SECONDS - EPSILON <= end <= event_time + CORE_END_AFTER_EVENT_SECONDS + EPSILON
    if contains_event or end_near_event:
        return 'positive_core', 1.0, 1.0, True
    if alert_time <= end < event_time:
        return 'positive_context_ignored', np.nan, CONTEXT_SOFT_LABEL, False
    if end <= alert_time - ALERT_MARGIN_SECONDS:
        return 'in_video_negative', 0.0, 0.0, True
    if start >= event_time + AFTERMATH_IGNORE_SECONDS:
        return 'post_event_ignored', np.nan, np.nan, False
    return 'transition_ignored', np.nan, np.nan, False

rows = []
for _, video in v3_video_manifest.sort_values('video_id', key=lambda values: values.astype(int)).iterrows():
    starts = sliding_starts(float(video['duration']), WINDOW_SECONDS, WINDOW_STRIDE_SECONDS)
    for window_index, start in enumerate(starts):
        end = round(min(float(video['duration']), start + WINDOW_SECONDS), 6)
        if int(video['label']) == 1:
            role, hard_label, soft_label, eligible = role_for_positive_window(start, end, float(video['time_of_event']), float(video['time_of_alert']))
        else:
            role, hard_label, soft_label, eligible = 'negative_video', 0.0, 0.0, True
        rows.append({
            'sequence_id': 'V3-SW-{}-w{:03d}'.format(video['video_id'], window_index),
            'video_id': str(video['video_id']),
            'video_path': video['video_path'],
            'split': video['split'],
            'outer_fold': int(video['outer_fold']),
            'video_label': int(video['label']),
            'time_of_event': video['time_of_event'],
            'time_of_alert': video['time_of_alert'],
            'duration': float(video['duration']),
            'window_index': int(window_index),
            'window_start': start,
            'window_end': end,
            'window_length': round(end - start, 6),
            'relative_event_position': video['relative_event_position'],
            'distance_to_event': np.nan if int(video['label']) == 0 else round(end - float(video['time_of_event']), 6),
            'distance_to_alert': np.nan if int(video['label']) == 0 else round(end - float(video['time_of_alert']), 6),
            'window_role': role,
            'label_policy': 'core_hard__context_ignore__safe_in_video_negative__negative_video',
            'hard_label': hard_label,
            'soft_label': soft_label,
            'eligible_for_base_training': bool(eligible),
            'eligible_for_mil_bag': True,
            'is_hard_negative': False,
            'source_model': '',
            'sampling_seed': REVIEW_SEED,
            'num_frames': NUM_FRAMES,
            'preprocessing_version': 'pending_v3_frame_cache',
            'weather': video['weather'],
            'light_conditions': video['light_conditions'],
            'scene': video['scene']
        })

sequence_manifest = pd.DataFrame(rows)
assert sequence_manifest['sequence_id'].is_unique
assert sequence_manifest['window_start'].ge(0.0).all()
assert sequence_manifest['window_end'].le(sequence_manifest['duration'] + EPSILON).all()
assert sequence_manifest['window_length'].sub(WINDOW_SECONDS).abs().le(EPSILON).all()
assert sequence_manifest.groupby('video_id')['split'].nunique().eq(1).all()
positive_core_count = sequence_manifest.loc[sequence_manifest['window_role'].eq('positive_core')].groupby('video_id').size()
assert set(positive_core_count.index) == set(v3_video_manifest.loc[v3_video_manifest['label'].eq(1), 'video_id'])
assert positive_core_count.ge(1).all()
print({'sliding_windows': len(sequence_manifest), 'positive_core_windows': int(sequence_manifest['window_role'].eq('positive_core').sum()), 'eligible_base_windows': int(sequence_manifest['eligible_for_base_training'].sum())})


{'sliding_windows': 8907, 'positive_core_windows': 757, 'eligible_base_windows': 6717}


In [4]:
# 3) Sample weights, MIL bags and deterministic review queues
eligible_counts = sequence_manifest.loc[sequence_manifest['eligible_for_base_training']].groupby('video_id').size().rename('eligible_windows_per_video')
sequence_manifest = sequence_manifest.merge(eligible_counts, on='video_id', how='left')
sequence_manifest['sample_weight'] = np.where(sequence_manifest['eligible_for_base_training'], 1.0 / sequence_manifest['eligible_windows_per_video'], 0.0)
sequence_manifest['sample_weight'] = sequence_manifest['sample_weight'].fillna(0.0)
assert np.isclose(sequence_manifest.groupby('video_id')['sample_weight'].sum(), 1.0).all()

SEQUENCE_MANIFEST_PATH = MANIFEST_ROOT / 'sequence_manifest_v3_sliding.csv'
sequence_manifest.sort_values(['split', 'video_id', 'window_index'], key=lambda values: values.astype(int) if values.name == 'video_id' else values).to_csv(SEQUENCE_MANIFEST_PATH, index=False)

bag_rows = []
for video_id, group in sequence_manifest.groupby('video_id', sort=False):
    first = group.iloc[0]
    bag_rows.append({
        'bag_id': 'V3-BAG-{}'.format(video_id),
        'video_id': video_id,
        'video_path': first['video_path'],
        'split': first['split'],
        'outer_fold': int(first['outer_fold']),
        'video_label': int(first['video_label']),
        'duration': float(first['duration']),
        'bag_window_count': int(len(group)),
        'positive_core_window_count': int(group['window_role'].eq('positive_core').sum()),
        'eligible_base_window_count': int(group['eligible_for_base_training'].sum()),
        'sequence_ids_json': json.dumps(group.sort_values('window_index')['sequence_id'].tolist()),
        'sampling_seed': REVIEW_SEED,
        'label_policy': 'video_level_mil_label_only'
    })
bag_manifest = pd.DataFrame(bag_rows)
assert len(bag_manifest) == 600 and bag_manifest['bag_id'].is_unique
assert bag_manifest.loc[bag_manifest['video_label'].eq(1), 'positive_core_window_count'].ge(1).all()
BAG_MANIFEST_PATH = MANIFEST_ROOT / 'bag_manifest_v3_mil.csv'
bag_manifest.sort_values(['split', 'video_id'], key=lambda values: values.astype(int) if values.name == 'video_id' else values).to_csv(BAG_MANIFEST_PATH, index=False)

positive_review = sequence_manifest.loc[sequence_manifest['window_role'].eq('positive_core')].sample(n=min(REVIEW_WINDOWS_PER_ROLE, int(sequence_manifest['window_role'].eq('positive_core').sum())), random_state=REVIEW_SEED).copy()
negative_review = sequence_manifest.loc[(sequence_manifest['split'].eq('train')) & (sequence_manifest['window_role'].eq('negative_video'))].groupby('video_id', group_keys=False).head(1).sample(n=REVIEW_WINDOWS_PER_ROLE, random_state=REVIEW_SEED).copy()
positive_review['review_group'] = 'positive_core'
negative_review['review_group'] = 'negative_candidate__not_hard_negative_yet'
review_queue = pd.concat([positive_review, negative_review], ignore_index=True)
REVIEW_QUEUE_PATH = REPORT_ROOT / 'window_review_queue_v3.csv'
review_queue.sort_values(['review_group', 'video_id', 'window_index'], key=lambda values: values.astype(int) if values.name == 'video_id' else values).to_csv(REVIEW_QUEUE_PATH, index=False)

display(sequence_manifest.groupby(['split', 'window_role']).size().rename('windows').reset_index())
print('Sequence manifest:', SEQUENCE_MANIFEST_PATH)
print('MIL bag manifest:', BAG_MANIFEST_PATH)
print('Review queue:', REVIEW_QUEUE_PATH)


,split,window_role,windows
0,train,in_video_negative,1286
1,train,negative_video,3497
2,train,positive_context_ignored,40
3,train,positive_core,603
4,train,post_event_ignored,1367
5,train,transition_ignored,346
6,validation,in_video_negative,311
7,validation,negative_video,866
8,validation,positive_context_ignored,8
9,validation,positive_core,154


Sequence manifest: P:\NexarCollisionData\manifests_v3\sequence_manifest_v3_sliding.csv
MIL bag manifest: P:\NexarCollisionData\manifests_v3\bag_manifest_v3_mil.csv
Review queue: P:\NexarCollisionData\reports_v3\window_review_queue_v3.csv


In [5]:
# 4) Distribution report, label policy and V3 registry
role_distribution = sequence_manifest.groupby(['split', 'video_label', 'window_role', 'eligible_for_base_training']).size().rename('windows').reset_index()
video_distribution = bag_manifest.groupby(['split', 'video_label']).agg(videos=('video_id', 'size'), mean_windows_per_video=('bag_window_count', 'mean'), min_windows_per_video=('bag_window_count', 'min'), max_windows_per_video=('bag_window_count', 'max')).reset_index()
distribution_report = pd.concat([
    role_distribution.assign(report_section='window_roles'),
    video_distribution.assign(report_section='video_bags', window_role='', eligible_for_base_training=np.nan, windows=np.nan)
], ignore_index=True, sort=False)
DISTRIBUTION_REPORT_PATH = REPORT_ROOT / 'window_distribution_report_v3.csv'
distribution_report.to_csv(DISTRIBUTION_REPORT_PATH, index=False)

POLICY_PATH = REPORT_ROOT / 'window_label_policy_v3.md'
policy_lines = [
    '# V3 sliding-window label policy',
    '',
    'Window length is 5 seconds with a deterministic 2.5-second stride.',
    '',
    '## Positive videos',
    '- positive_core: event is inside the window, or window end is in [event - 1.5s, event + 1.0s]. hard_label=1 and eligible for base training.',
    '- positive_context_ignored: window ends from alert to event but is not core. It receives soft_label=0.7 for a later explicit ablation, but is ignored in the initial hard-label baseline.',
    '- in_video_negative: window end is at least 0.5s before alert. hard_label=0 and eligible for base training.',
    '- transition and post-event windows: ignored initially to avoid ambiguous pre-risk and aftermath labels.',
    '',
    '## Negative videos',
    '- Every sliding window is negative_video with hard_label=0. Per-video sample weights sum to one so long videos cannot dominate training.',
    '',
    '## Safety and leakage rules',
    '- time_of_event and time_of_alert exist only in manifest generation, analysis and review. They are forbidden model inputs.',
    '- Hard negatives do not exist yet. V3-3 will score only negative training windows and annotate selected rows without changing validation.',
    '- Validation is deterministic. Full-MP4 inference remains all windows plus video-level aggregation.'
]
POLICY_PATH.write_text('\n'.join(policy_lines) + '\n', encoding='utf-8')

summary = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'video_manifest_v3': str(VIDEO_MANIFEST_V3_PATH),
    'sequence_manifest_v3_sliding': str(SEQUENCE_MANIFEST_PATH),
    'bag_manifest_v3_mil': str(BAG_MANIFEST_PATH),
    'videos': int(len(v3_video_manifest)),
    'sliding_windows': int(len(sequence_manifest)),
    'mil_bags': int(len(bag_manifest)),
    'positive_core_windows': int(sequence_manifest['window_role'].eq('positive_core').sum()),
    'base_training_windows': int(sequence_manifest['eligible_for_base_training'].sum()),
    'window_seconds': WINDOW_SECONDS,
    'window_stride_seconds': WINDOW_STRIDE_SECONDS,
    'num_frames': NUM_FRAMES,
    'validation_is_deterministic': True,
    'hard_negative_status': 'pending_v3_3',
    'source_video_manifest_sha256': sha256_file(VIDEO_MANIFEST_V2_PATH),
    'source_cv_folds_sha256': sha256_file(CV_FOLDS_PATH)
}
SUMMARY_PATH = REPORT_ROOT / 'sequence_manifest_v3_sliding_summary.json'
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

registry = pd.read_csv(REGISTRY_PATH)
dataset_row = {'run_id': 'V3_02_SLIDING_WINDOW_DATASET', 'stage': 'V3-2 sliding_aligned_sampling', 'model_id': 'none', 'dataset_version': 'v3_sliding_core_context_v1', 'split_version': 'metadata_split_v1_and_cv_folds_v3', 'window_version': '5s_stride2.5s_core_context_v1', 'feature_version': 'pending_v3_frame_cache', 'augmentation_version': 'none', 'checkpoint_path': '', 'config_path': 'notebooks/32_v3_window_dataset.ipynb', 'git_commit': 'not_available', 'status': 'completed', 'primary_metric': 'data_tests', 'primary_value': 1.0, 'notes': 'Conservative core/context/in-video-negative policy. Hard-negative mining pending V3-3.'}
registry = registry.loc[~registry['run_id'].eq('V3_02_SLIDING_WINDOW_DATASET')]
registry = pd.concat([registry, pd.DataFrame([dataset_row])], ignore_index=True)
registry.to_csv(REGISTRY_PATH, index=False)

display(role_distribution)
display(video_distribution)
print('Policy:', POLICY_PATH)
print('Summary:', SUMMARY_PATH)


,split,video_label,window_role,eligible_for_base_training,windows
0,train,0,negative_video,True,3497
1,train,1,in_video_negative,True,1286
2,train,1,positive_context_ignored,False,40
3,train,1,positive_core,True,603
4,train,1,post_event_ignored,False,1367
5,train,1,transition_ignored,False,346
6,validation,0,negative_video,True,866
7,validation,1,in_video_negative,True,311
8,validation,1,positive_context_ignored,False,8
9,validation,1,positive_core,True,154


,split,video_label,videos,mean_windows_per_video,min_windows_per_video,max_windows_per_video
0,train,0,240,14.570833,7,23
1,train,1,240,15.175000,7,23
2,validation,0,60,14.433333,5,16
3,validation,1,60,15.033333,7,16


Policy: P:\NexarCollisionData\reports_v3\window_label_policy_v3.md
Summary: P:\NexarCollisionData\reports_v3\sequence_manifest_v3_sliding_summary.json
